### Workshop the steps in a synthetic reconstruction of narG variants. 

First, build two faa's, one to be used as a reference, the other to generate raw reads.

In [5]:
import numpy as np
ref_directory = '../data/whole_genomes/K00370_rep.faa'

from Bio import SeqIO
import random

# Read all sequences
records = list(SeqIO.parse(ref_directory, "fasta"))
print(f"Total sequences: {len(records)}")

# Shuffle and split 70/30
random.seed(42) #I'm fixing a seed here
random.shuffle(records) 
split_point = int(0.89 * len(records)) #0.89 makes it so there are exactly 100 spike-in sequences

reference_set = records[:split_point]
spikein_set = records[split_point:]

# Write outputs
SeqIO.write(reference_set, "../out/reconstruction/reference.faa", "fasta")
SeqIO.write(spikein_set, "../out/reconstruction/spikein.faa", "fasta")

print(f"Reference: {len(reference_set)} sequences")
print(f"Spike-in: {len(spikein_set)} sequences")

Total sequences: 906
Reference: 806 sequences
Spike-in: 100 sequences


Generate raw reads

In [ ]:
from Bio import SeqIO
import random

def back_translate(protein_seq):
    """Convert protein sequence to DNA using common codons"""
    codon_table = {
        'A': 'GCT', 'R': 'CGT', 'N': 'AAT', 'D': 'GAT',
        'C': 'TGT', 'Q': 'CAG', 'E': 'GAA', 'G': 'GGT',
        'H': 'CAT', 'I': 'ATT', 'L': 'CTG', 'K': 'AAA',
        'M': 'ATG', 'F': 'TTT', 'P': 'CCT', 'S': 'TCT',
        'T': 'ACT', 'W': 'TGG', 'Y': 'TAT', 'V': 'GTT',
        '*': 'TAA'
    }
    dna_seq = ''
    for aa in protein_seq:
        dna_seq += codon_table.get(aa, 'NNN')  # Use NNN for unknown amino acids
    return dna_seq

def add_sequencing_errors(read, error_rate=0.001):
    """Add simple substitution errors to simulate sequencing errors"""
    bases = ['A', 'T', 'C', 'G']
    result = []
    for base in read:
        if random.random() < error_rate:
            # Choose a different base
            new_base = random.choice([b for b in bases if b != base])
            result.append(new_base)
        else:
            result.append(base)
    return ''.join(result)

def generate_paired_end_reads(dna_sequence, read_length=150, coverage=30, insert_size=300, error_rate=0.001):
    """Generate paired-end reads with proper insert size"""
    reads_r1 = []
    reads_r2 = []
    seq_length = len(dna_sequence)
    
    # Calculate number of read pairs needed
    num_pairs = (seq_length * coverage) // (2 * read_length)
    
    for _ in range(num_pairs):
        # Random start position for R1
        start_r1 = random.randint(0, seq_length - insert_size)
        end_r1 = start_r1 + read_length
        start_r2 = start_r1 + insert_size - read_length
        
        # Ensure we don't go beyond sequence boundaries
        if start_r2 + read_length <= seq_length:
            read_r1 = dna_sequence[start_r1:end_r1]
            read_r2 = dna_sequence[start_r2:start_r2 + read_length]
            
            # Add seequencing errors
            read_r1 = add_sequencing_errors(read_r1, error_rate = error_rate)
            read_r2 = add_sequencing_errors(read_r2, error_rate = error_rate)
            
            # R2 is reverse complement (simulating actual sequencing)
            read_r2 = reverse_complement(read_r2)
            
            reads_r1.append(read_r1)
            reads_r2.append(read_r2)
    
    return reads_r1, reads_r2

def reverse_complement(seq):
    """Simple reverse complement"""
    comp = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
    return ''.join(comp.get(base, base) for base in reversed(seq))

# Read spike-in protein sequences
spikein_records = list(SeqIO.parse("../out/reconstruction/spikein.faa", "fasta"))


#Generate many raw reads in a loop
coverage_sweep = np.linspace(5, 100, 10)
error_sweep = [0, 0.001, 0.005]
for cov in coverage_sweep:
    for err in error_sweep:
        # Generate paired-end DNA reads
        all_r1 = []
        all_r2 = []

        for record in spikein_records:
            # Convert protein to DNA first!
            dna_sequence = back_translate(str(record.seq))
            print(f"Converted {record.id}: {len(record.seq)}aa -> {len(dna_sequence)}bp DNA")
            
            r1, r2 = generate_paired_end_reads(dna_sequence, read_length=150, coverage = cov, error_rate = err)
            all_r1.extend(r1)
            all_r2.extend(r2)

        print(f"Generated {len(all_r1)} DNA read pairs")

        # Write R1 and R2 files as FASTA
        with open(f"../out/reconstruction/raw_reads/spikein_R1_{cov}_{err}.fasta", "w") as f1, \
            open(f"../out/reconstruction/raw_reads/spikein_R2_{cov}_{err}.fasta", "w") as f2:
            
            for i, (read1, read2) in enumerate(zip(all_r1, all_r2)):
                # R1 file - FASTA format
                f1.write(f">read_{i}/1\n")
                f1.write(f"{read1}\n")
                
                # R2 file - FASTA format  
                f2.write(f">read_{i}/2\n")
                f2.write(f"{read2}\n")

        print("Paired-end DNA reads written to:")
        print(f"  ../out/reconstruction/raw_reads/spikein_R1_{cov}_{err}.fasta")
        print(f"  ../out/reconstruction/raw_reads/spikein_R2_{cov}_{err}.fasta")

### Then we moved to QUEST to generate contigs and ORFs from these raw reads. 

Analysis of those results is below. 

In [3]:
from Bio import SeqIO
from Bio.Align import PairwiseAligner
import numpy as np

def assess_reconstruction(spikein_faa, prodigal_faa, identity_threshold=0.95):
    """
    Assess how many spike-in variants were successfully reconstructed
    """
    # Load sequences
    spikein_seqs = {rec.id: str(rec.seq) for rec in SeqIO.parse(spikein_faa, "fasta")}
    prodigal_seqs = {rec.id: str(rec.seq) for rec in SeqIO.parse(prodigal_faa, "fasta")}
    
    print(f"=== Reconstruction Assessment ===")
    print(f"Spike-in variants: {len(spikein_seqs)}")
    print(f"Reconstructed ORFs: {len(prodigal_seqs)}")
    
    # Simple alignment setup
    aligner = PairwiseAligner()
    aligner.mode = 'global'
    aligner.match_score = 2
    aligner.mismatch_score = -1
    aligner.open_gap_score = -0.5
    aligner.extend_gap_score = -0.1
    
    # Track matches
    reconstructed_variants = set()
    reconstruction_details = []
    
    # For each spike-in variant, find best match in reconstructed ORFs
    for spike_id, spike_seq in spikein_seqs.items():
        best_match_id = None
        best_identity = 0
        
        for orf_id, orf_seq in prodigal_seqs.items():
            # Quick length filter
            if abs(len(spike_seq) - len(orf_seq)) > 100:  # Allow some length variation
                continue
                
            alignment = aligner.align(spike_seq, orf_seq)[0]
            identity = alignment.score / max(len(spike_seq), len(orf_seq))
            
            if identity > best_identity:
                best_identity = identity
                best_match_id = orf_id
        
        if best_identity >= identity_threshold:
            reconstructed_variants.add(spike_id)
            status = "RECONSTRUCTED"
        else:
            status = "MISSING"
            
        reconstruction_details.append({
            'spike_id': spike_id,
            'best_match': best_match_id,
            'identity': best_identity,
            'status': status
        })
    
    # Print results
    print(f"\n=== Results ===")
    print(f"Successfully reconstructed: {len(reconstructed_variants)}/{len(spikein_seqs)} variants")
    print(f"Reconstruction rate: {len(reconstructed_variants)/len(spikein_seqs)*100:.1f}%")
    
    # Show details
    print(f"\n=== Reconstruction Details ===")
    for detail in sorted(reconstruction_details, key=lambda x: x['identity'], reverse=True):
        print(f"{detail['spike_id']} -> {detail['best_match']}: {detail['identity']:.3f} ({detail['status']})")
    
    return reconstruction_details

# Run the assessment
results = assess_reconstruction(
    spikein_faa="../out/reconstruction/spikein.faa",
    prodigal_faa="../out/reconstruction/prodigal_orfs.faa", 
    identity_threshold=0.95  # 95% identity threshold
)

=== Reconstruction Assessment ===
Spike-in variants: 100
Reconstructed ORFs: 303

=== Results ===
Successfully reconstructed: 69/100 variants
Reconstruction rate: 69.0%

=== Reconstruction Details ===
tni:TVNIR_1119 -> NODE_6_length_3493_cov_9.671321_1: 1.994 (RECONSTRUCTED)
dap:Dacet_2070 -> NODE_4_length_3505_cov_9.844058_1: 1.936 (RECONSTRUCTED)
bts:Btus_1655 -> NODE_3_length_3513_cov_12.775882_1: 1.891 (RECONSTRUCTED)
afx:JZ786_13520 -> NODE_8_length_3475_cov_12.375731_1: 1.867 (RECONSTRUCTED)
afk:ACNAN0_11500 -> NODE_15_length_3375_cov_10.768072_1: 1.823 (RECONSTRUCTED)
nnv:QNH39_14340 -> NODE_9_length_3465_cov_20.192082_1: 1.508 (RECONSTRUCTED)
spet:CEP67_06030 -> NODE_12_length_3405_cov_10.727761_1: 1.394 (RECONSTRUCTED)
vim:GWK91_05660 -> NODE_9_length_3465_cov_20.192082_1: 1.266 (RECONSTRUCTED)
gym:GYMC10_1981 -> NODE_12_length_3405_cov_10.727761_1: 1.255 (RECONSTRUCTED)
saca:FFV09_21985 -> NODE_12_length_3405_cov_10.727761_1: 1.237 (RECONSTRUCTED)
lalw:BTM29_08930 -> NODE_15_